# 02 — Preprocessing: Очистка и подготовка данных


In [20]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

TEXT_COL = "text" 
LABEL_COL = "label"

## 1. Загрузка сырых данных

In [21]:
dataset = load_dataset("ai-forever/ru-reviews-classification")

df_train = dataset["train"].to_pandas()
df_test  = dataset["test"].to_pandas()

print(f"Train: {len(df_train)}, Test (holdout): {len(df_test)}")
print(f"Колонки: {df_train.columns.tolist()}")

Train: 45000, Test (holdout): 15000
Колонки: ['text', 'label_text', 'label', 'id']


## 2. Полная очистка

In [22]:
def clean_data(df: pd.DataFrame, text_col: str, label_col: str) -> pd.DataFrame:
    original_len = len(df)
    
    # 1. Дубликаты по тексту
    before = len(df)
    df = df.drop_duplicates(subset=[text_col])
    print(f"После дедупликации: {len(df)} (убрано {before - len(df)})")
    
    # 2. Пустые тексты после стрипа
    df[text_col] = df[text_col].str.strip()
    before = len(df)
    df = df[df[text_col].str.len() > 0]
    print(f"После удаления пустых: {len(df)} (убрано {before - len(df)})")
    
    # 3. Очень короткие тексты (меньше 3 слов) — скорее всего мусор
    before = len(df)
    df = df[df[text_col].str.split().str.len() >= 3]
    print(f"После удаления слишком коротких (< 3 слов): {len(df)} (убрано {before - len(df)})")
    
    df = df.reset_index(drop=True)
    return df


df_train_clean = clean_data(df_train, TEXT_COL, LABEL_COL)
df_test_clean  = clean_data(df_test,  TEXT_COL, LABEL_COL)

После дедупликации: 45000 (убрано 0)
После удаления пустых: 45000 (убрано 0)
После удаления слишком коротких (< 3 слов): 43652 (убрано 1348)
После дедупликации: 15000 (убрано 0)
После удаления пустых: 15000 (убрано 0)
После удаления слишком коротких (< 3 слов): 14540 (убрано 460)


## 3. Нормализация текста(датасет уже нормализирован)

In [23]:
def normalize_text(text: str) -> str:
    # Убираем HTML-теги
    text = re.sub(r"<[^>]+>", " ", text)
    # Убираем URL
    text = re.sub(r"https?://\S+", " ", text)
    # Убираем повторяющиеся пробелы
    text = re.sub(r"\s+", " ", text)
    # Lower
    text = text.lower().strip()
    return text


df_train_clean["text_clean"] = df_train_clean[TEXT_COL].apply(normalize_text)
df_test_clean["text_clean"]  = df_test_clean[TEXT_COL].apply(normalize_text)

#Проверка 
print("До:",  df_train_clean[TEXT_COL].iloc[0])
print("После:", df_train_clean["text_clean"].iloc[0])

До: всё пришло спасибо. только немного короче чем я ожидала
так всё супер
После: всё пришло спасибо. только немного короче чем я ожидала так всё супер


## 4. Feature Engineering

In [24]:
def add_features(df: pd.DataFrame, text_col: str = "text_clean") -> pd.DataFrame:
    # Длина текста
    df["feat_char_len"] = df[text_col].str.len()
    df["feat_word_count"] = df[text_col].str.split().str.len()
    
    # Доля заглавных букв в оригинале (сигнал эмоциональности)
    df["feat_upper_ratio"] = df[TEXT_COL].apply(
        lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1)
    )
    
    # Количество восклицательных и вопросительных знаков
    df["feat_exclamation_count"] = df[TEXT_COL].str.count("!")
    df["feat_question_count"]    = df[TEXT_COL].str.count(r"\?")
    
    # Кол-во эмодзи
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U00002702-\U000027B0"
        "]+", flags=re.UNICODE
    )
    df["feat_emoji_count"] = df[TEXT_COL].apply(lambda t: len(emoji_pattern.findall(t)))
    
    # Средняя длина слова
    df["feat_avg_word_len"] = df[text_col].apply(
        lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0
    )
    
    return df


df_train_clean = add_features(df_train_clean)
df_test_clean  = add_features(df_test_clean)

feature_cols = [c for c in df_train_clean.columns if c.startswith("feat_")]
print(f"Исходных фич: 1 (text)")
print(f"Новых числовых фич: {len(feature_cols)}")
print(f"Фичи: {feature_cols}")
df_train_clean[feature_cols].describe()

Исходных фич: 1 (text)
Новых числовых фич: 7
Фичи: ['feat_char_len', 'feat_word_count', 'feat_upper_ratio', 'feat_exclamation_count', 'feat_question_count', 'feat_emoji_count', 'feat_avg_word_len']


,feat_char_len,feat_word_count,feat_upper_ratio,feat_exclamation_count,feat_question_count,feat_emoji_count,feat_avg_word_len
count,43652.000000,43652.000000,43652.000000,43652.000000,43652.000000,43652.000000,43652.000000
mean,136.448112,21.079607,0.023973,0.704412,0.026299,0.000504,5.735195
std,126.049419,19.946659,0.053717,2.154461,0.278835,0.026211,1.150264
min,8.000000,3.000000,0.000000,0.000000,0.000000,0.000000,2.000000
25%,57.000000,9.000000,0.000000,0.000000,0.000000,0.000000,5.000000
50%,97.000000,15.000000,0.017857,0.000000,0.000000,0.000000,5.550000
75%,170.000000,26.000000,0.030612,0.000000,0.000000,0.000000,6.222222
max,1000.000000,192.000000,0.916667,77.000000,13.000000,3.000000,25.000000


## 5. Выбросы

In [25]:
import matplotlib.pyplot as plt

# Смотрим на выбросы по длине текста
q99 = df_train_clean["feat_word_count"].quantile(0.99)
print(f"99-й перцентиль по кол-ву слов: {q99}")

outliers = df_train_clean[df_train_clean["feat_word_count"] > q99]
print(f"Выбросов (>99p): {len(outliers)} ({len(outliers)/len(df_train_clean)*100:.1f}%)")


99-й перцентиль по кол-ву слов: 104.0
Выбросов (>99p): 427 (1.0%)


Решение: выбросы по длине оставляем — трансформер обрежет до 512 токенов автоматически

## 6. Train / Val / Test сплит

In [26]:
df_tr, df_val = train_test_split(
    df_train_clean,
    test_size=0.15,
    random_state=SEED,
    stratify=df_train_clean[LABEL_COL]
)

print(f"Train:      {len(df_tr)} примеров")
print(f"Validation: {len(df_val)} примеров")
print(f"Test:       {len(df_test_clean)} примеров")
print(f"\ntrain:\n{df_tr[LABEL_COL].value_counts(normalize=True)}")
print(f"\nval:\n{df_val[LABEL_COL].value_counts(normalize=True)}")

Train:      37104 примеров
Validation: 6548 примеров
Test:       14540 примеров

train:
label
0    0.339613
1    0.331824
2    0.328563
Name: proportion, dtype: float64

val:
label
0    0.339646
1    0.331857
2    0.328497
Name: proportion, dtype: float64


## 7. Сохранение

In [27]:
import os

OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

df_tr.to_csv(f"{OUT_DIR}/train.csv", index=False)
df_val.to_csv(f"{OUT_DIR}/val.csv",   index=False)
df_test_clean.to_csv(f"{OUT_DIR}/test.csv", index=False)

print("OK")
for name in ["train.csv", "val.csv", "test.csv"]:
    path = f"{OUT_DIR}/{name}"
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f"  {name}: {size_mb:.2f} MB")

OK
  train.csv: 19.48 MB
  val.csv: 3.45 MB
  test.csv: 7.65 MB


## 8. Итоговое резюме

In [28]:
print(f"Исходных фич:  1 (text)")
print(f"Новых фич:     {len(feature_cols)}")
print(f"Train:         {len(df_tr)}")
print(f"Val:           {len(df_val)}")
print(f"Test holdout:  {len(df_test_clean)}")
print(f"Seed:          {SEED}")
print(f"Метрика:       Macro F1")

Исходных фич:  1 (text)
Новых фич:     7
Train:         37104
Val:           6548
Test holdout:  14540
Seed:          42
Метрика:       Macro F1
